In [1]:
import cv2
import torch
import logging
from datetime import datetime
import pygame
import smtplib
from email.mime.text import MIMEText
import time
import csv
import os
import threading

pygame 2.6.1 (SDL 2.28.4, Python 3.10.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Sender information
sender_email = "jerryphung282@gmail.com"
app_password = "owjnsrmjkjtajvxa"   # 16 digit app password(created in Google Account Security)

# Receiver information
receiver_email = "phunggiahuy17@gmail.com"

In [3]:
# Create folders to save captures and videos
image_folder = "captures"
video_folder = "videos"
os.makedirs(image_folder, exist_ok=True)
os.makedirs(video_folder, exist_ok=True)

In [4]:
# Video writer
video_writer = None
recording = False
video_fps = 20
video_codec = cv2.VideoWriter_fourcc(*'XVID')  # hoặc 'mp4v' nếu lưu .mp4
video_filename = None

In [5]:
alert_triggered = False
pygame.mixer.init()
pygame.mixer.music.load("alarm.mp3")

In [6]:
logging.basicConfig(
    filename='smoking_detections.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [ ]:
# Load the pre-trained model
# If you want to try with good performance: C:\Users\admin\OneDrive\Desktop\smoking-detection\training\epochs_100\weights.pt
# If you just want to try mine: C:\Users\admin\OneDrive\Desktop\smoking-detection\training\epochs_50\best.pt
model = torch.hub.load('ultralytics/yolov5', 'custom', path=r'C:\Users\admin\OneDrive\Desktop\smoking-detection\training\epochs_50\best.pt')

Using cache found in C:\Users\admin/.cache\torch\hub\ultralytics_yolov5_master
c:\Users\admin\anaconda3\envs\smoke-detect-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
YOLOv5  2025-4-30 Python-3.10.16 torch-2.1.0+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [8]:
def send_email_alert(subject, body):
    msg = MIMEText(body)
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = receiver_email

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(sender_email, app_password)
            server.send_message(msg)
            print("Alert email has been sent.")
    except Exception as e:
        print(f"Error sending email: {e}")


In [9]:
# Function to send email in a separate thread
def async_send_email(subject, body):
    threading.Thread(target=send_email_alert, args=(subject, body)).start()

# Function to play sound in a separate thread
def async_play_sound():
    threading.Thread(target=pygame.mixer.music.play).start()

In [10]:
# Set up logging
csv_log_path = 'detection_log.csv'
if not os.path.exists(csv_log_path):
    with open(csv_log_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'class_name', 'confidence'])

In [13]:
# Display window
window_name = 'Smoking Detection'
cap = cv2.VideoCapture(0)

# Cooldown timers
email_cooldown = 30  # seconds
sound_cooldown = 10  # seconds

last_email_time = 0
last_sound_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    annotated_frame = results.render()[0]
    cv2.imshow(window_name, annotated_frame)

    detections = results.pandas().xyxy[0]
    smoking_detected = False

    for _, row in detections.iterrows():
        class_name = row['name']
        confidence = row['confidence']

        if class_name.lower() in ['smoke', 'smoking', 'cigarette']:
            smoking_detected = True
            logging.info(f"Detected {class_name} with confidence {confidence:.2f}")
            current_time = time.time()

            # Play sound alert
            if current_time - last_sound_time > sound_cooldown:
                async_play_sound()
                last_sound_time = current_time

            # Send email alert
            if current_time - last_email_time > email_cooldown:
                async_send_email(
                    subject="Smoking Detected",
                    body=f"Detected {class_name} with confidence {confidence:.2f}!"
                )
                last_email_time = current_time

            # Log to CSV
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            with open(csv_log_path, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, class_name, f"{confidence:.2f}"])

            # Save image
            timestamp_img = datetime.now().strftime('%Y%m%d_%H%M%S')
            img_path = os.path.join(image_folder, f"smoke_{timestamp_img}.jpg")
            cv2.imwrite(img_path, annotated_frame)
            logging.info(f"Saved image: {img_path}")

            # Start video recording
            if not recording:
                video_filename = os.path.join(video_folder, f"smoke_{timestamp_img}.mp4")
                height, width = frame.shape[:2]
                video_writer = cv2.VideoWriter(video_filename, video_codec, video_fps, (width, height))
                recording = True
                logging.info(f"Started recording: {video_filename}")

            break  # only trigger once per frame

    # If recording, continue saving frames
    if recording:
        video_writer.write(annotated_frame)
        if not smoking_detected:
            video_writer.release()
            logging.info(f"Stopped recording: {video_filename}")
            recording = False

    # Exit if 'q' is pressed or window closed
    if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
        break
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
if recording:
    video_writer.release()
cv2.destroyAllWindows()

Alert email has been sent.
